# Étape 3 – Scraping des hôtels sur Booking.com

**Objectif** : pour chaque destination, récupérer des informations sur les hôtels. L'énoncé suggère au minimum :
nom, URL de la page Booking, coordonnées GPS (latitude, longitude), note des utilisateurs, description.

**Outil** : [Scrapy](https://docs.scrapy.org/) (vu en formation, module Web Scraping) et ses sélecteurs XPath/CSS.

**Compétence du référentiel** : C3 – collecter des données du Web *via des librairies de type Scrapy ou BeautifulSoup,
dans le respect des normes de protection des données (RGPD)*.

**Cadre éthique et RGPD** :
- seules des **données d'établissements** (informations commerciales publiques) sont collectées :
  **aucune donnée personnelle** (pas de noms ni de textes d'avis de clients) ;
- **charge minimale** sur le site : quelques dizaines de pages, avec une pause entre deux requêtes ;
- usage **strictement pédagogique**, imposé par l'énoncé de la certification. Les conditions d'utilisation de Booking
  n'autorisent pas la collecte automatisée : en contexte professionnel, il faudrait passer par une source autorisée
  (API partenaire, accord de données).

**Démarche en deux temps** (comme pour la météo) :
1. **Test sur une seule destination** : la page est-elle accessible ? quelles informations peut-on en extraire ?
2. Ensuite seulement : le spider Scrapy complet pour les 35 destinations.

## 1. Imports et chemins

In [1]:
import re
from pathlib import Path
from urllib.parse import urlencode, urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from scrapy import Selector  # le sélecteur de Scrapy, utilisable sans lancer de spider


def find_project_root(start: Path) -> Path:
    """Remonte l'arborescence jusqu'au dossier qui contient requirements.txt (racine du projet)."""
    for folder in [start, *start.parents]:
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("Racine du projet introuvable (requirements.txt absent)")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"

BASE_URL = "https://www.booking.com"
HEADERS = {
    # En-têtes d'un navigateur courant : sans eux, le site renvoie une page vide ou bloquée
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/130.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9",
}
TEST_CITY = "Collioure"  # n°1 du classement météo de l'étape 2

## 2. Que dit le fichier `robots.txt` de Booking ?

Le fichier `robots.txt` indique aux robots les pages que le site **souhaite** voir explorées ou non.
Il n'a pas de valeur juridique, mais le consulter fait partie des **bonnes pratiques** du scraping :
on sait ainsi en connaissance de cause si l'on s'écarte des souhaits du site.

In [2]:
robots_url = f"{BASE_URL}/robots.txt"
robots_lines = requests.get(robots_url, headers=HEADERS, timeout=15).text.splitlines()

parser = RobotFileParser()
parser.parse(robots_lines)

search_url = f"{BASE_URL}/searchresults.fr.html?" + urlencode({"ss": TEST_CITY})
print("Page de recherche autorisée par robots.txt :", parser.can_fetch(HEADERS["User-Agent"], search_url))
print("Page d'hôtel autorisée par robots.txt      :", parser.can_fetch(HEADERS["User-Agent"], f"{BASE_URL}/hotel/fr/exemple.fr.html"))

print("\nRègles concernant les recherches et les hôtels :")
for line in robots_lines:
    if re.search(r"searchresults|/hotel", line, flags=re.IGNORECASE):
        print(" ", line)

Page de recherche autorisée par robots.txt : True
Page d'hôtel autorisée par robots.txt      : True

Règles concernant les recherches et les hôtels :
  Disallow: /hotelsonmap.*.json
  Disallow: /hotelsonmap.*.json
  Disallow: /hotelfeaturedreviews/
  Disallow: /hotelfeaturedreviews/
  Disallow: /hotelsonmap.*.json
  Disallow: /hotel/us/the-airstream-van.*.html
  Disallow: /hotel_rt_onview
  Disallow: /hotelfeaturedreviews/
  Allow: /hotel/index.hi.*


## 3. Test : page de résultats de recherche pour une destination

On vérifie successivement :
1. le **code HTTP** et l'**URL finale** (une redirection peut signaler un blocage) ;
2. le **titre** de la page (une page de vérification anti-robot a un titre différent) ;
3. le **nombre de fiches d'hôtels** présentes dans le HTML (balises `data-testid="property-card"`).

La page est enregistrée dans `data/raw/` (exclu de Git) pour pouvoir l'analyser hors ligne.

In [3]:
params = {"ss": TEST_CITY, "lang": "fr", "order": "bayesian_review_score"}  # tri par note des clients
response = requests.get(f"{BASE_URL}/searchresults.fr.html", params=params, headers=HEADERS, timeout=30)

page = Selector(text=response.text)
cards = page.css('div[data-testid="property-card"]')

print("Code HTTP     :", response.status_code)
print("URL finale    :", response.url[:120])
print("Taille du HTML:", len(response.text), "caractères")
print("Titre         :", page.css("title::text").get())
print("Fiches d'hôtels trouvées :", len(cards))

SEARCH_TEST_FILE = RAW_DIR / "booking_test_search.html"
SEARCH_TEST_FILE.write_text(response.text, encoding="utf-8")
print("Page enregistrée :", SEARCH_TEST_FILE)

Code HTTP     : 202
URL finale    : https://www.booking.com/searchresults.fr.html?ss=Collioure&lang=fr&order=bayesian_review_score
Taille du HTML: 3962 caractères
Titre         : None
Fiches d'hôtels trouvées : 0
Page enregistrée : c:\Users\frede\OneDrive\Documents\_Jedha\2- dsfs-od-15 - Fullstack\Final Projects - dsfs-od-15\B1 - Kayak\kayak-plan-your-trip\data\raw\booking_test_search.html


## 4. Extraction des informations visibles sur chaque fiche

Les sélecteurs s'appuient sur les attributs `data-testid`, plus **stables** que les classes CSS
(les noms de classes de Booking sont générés automatiquement et changent régulièrement).

In [5]:
def first_number(text: str | None) -> float | None:
    """Extrait le premier nombre d'un texte ('Note : 8,7' -> 8.7), ou None."""
    match = re.search(r"\d+(?:[.,]\d+)?", text or "")
    return float(match.group().replace(",", ".")) if match else None


hotels_test = []
for card in cards:
    link = card.css('a[data-testid="title-link"]::attr(href)').get()
    hotels_test.append({
        "hotel_name": card.css('div[data-testid="title"]::text').get(),
        "hotel_url": urljoin(BASE_URL, link.split("?")[0]) if link else None,  # URL sans paramètres de suivi
        "review_score": first_number(" ".join(card.css('div[data-testid="review-score"] ::text').getall())),
        "address": card.css('span[data-testid="address"]::text').get(),
    })

hotels_test = pd.DataFrame(hotels_test)
print(f"{len(hotels_test)} hôtels extraits")
hotels_test.head(10)

0 hôtels extraits


""


## 5. Test : page détaillée d'un hôtel (coordonnées GPS et description)

Les coordonnées et la description ne figurent pas sur la page de résultats :
il faut ouvrir la page de chaque hôtel. On teste ici sur le premier hôtel de la liste.

In [6]:
if hotels_test.empty:
    print("Aucun hôtel extrait à l'étape précédente : test impossible.")
else:
    hotel_url = hotels_test.loc[0, "hotel_url"]
    hotel_response = requests.get(hotel_url, headers=HEADERS, timeout=30)
    hotel_page = Selector(text=hotel_response.text)

    # Coordonnées : attribut 'data-atlas-latlng' (format "lat,lon") du lien vers la carte
    latlng = hotel_page.css("[data-atlas-latlng]::attr(data-atlas-latlng)").get()
    # Description : bloc 'property-description'
    description = " ".join(hotel_page.css('[data-testid="property-description"] ::text').getall()).strip()

    print("Hôtel        :", hotels_test.loc[0, "hotel_name"])
    print("Code HTTP    :", hotel_response.status_code)
    print("Coordonnées  :", latlng)
    print("Description  :", (description[:300] + "...") if description else None)

    HOTEL_TEST_FILE = RAW_DIR / "booking_test_hotel.html"
    HOTEL_TEST_FILE.write_text(hotel_response.text, encoding="utf-8")
    print("Page enregistrée :", HOTEL_TEST_FILE)

Aucun hôtel extrait à l'étape précédente : test impossible.


## 6. Analyse du test et choix pour le spider complet

**Constat du test (sections 3 à 5)** : la page renvoyée n'est **pas** une liste d'hôtels mais une page de
**vérification anti-robot AWS WAF** (code HTTP **202**, page sans titre, script `challenge.js`, variable
`awsWafCookieDomainList`). Un navigateur réel exécute ce script, obtient un cookie et recharge la page ;
un client HTTP simple (`requests`, Scrapy seul) n'exécute pas de JavaScript et reste bloqué.
En revanche, `robots.txt` **autorise** les pages de recherche et les pages d'hôtels.

**Choix retenus pour le spider** (`scraping/booking_spider.py`) :

| Choix | Justification |
|---|---|
| **Scrapy + scrapy-playwright** : pages chargées par un navigateur Chromium réel | Exécute la vérification JavaScript comme pour un internaute, tout en gardant Scrapy (outil attendu par le référentiel) |
| Aucun camouflage : le navigateur envoie ses propres en-têtes, pas de rotation d'adresses IP ; arrêt en cas de CAPTCHA | Respect du site |
| 1 page à la fois, 3 s de pause entre deux pages | Charge minimale |
| Extraction depuis le **bloc JSON** intégré à la page (`data-capla-store-data="apollo"`) plutôt que depuis le HTML des fiches | Plus complet (coordonnées GPS, adresse, note, nombre d'avis, description) et plus stable : la mise en page HTML change régulièrement. Évite d'ouvrir la page de chaque hôtel (35 pages au lieu de 700+) |
| Filtre Booking **« hôtels » uniquement** (`nflt=ht_id=204`) | Fidélité à l'énoncé ; argument **RGPD** : l'adresse d'une location entre particuliers est souvent un domicile privé |
| Tri par **note pondérée par le nombre d'avis** (`order=bayesian_review_score`) | Un 9,9 sur 12 avis ne passe pas devant un 9,3 sur 650 avis |
| Champs ignorés : résumés d'avis, données des clients | Aucune donnée personnelle collectée |

## 7. Collecte complète (exécutée depuis le terminal)

Scrapy ne pouvant pas être relancé plusieurs fois dans un même notebook, le spider est un **script** lancé depuis
Anaconda PowerShell Prompt (environnement `kayak`, racine du projet) ; le notebook sert à **analyser** le résultat.

```powershell
scrapy runspider scraping/booking_spider.py -O data/raw/hotels_booking.json
```

Résultat obtenu : `item_scraped_count: 700` (20 hôtels × 35 destinations), aucune erreur.

## 8. Chargement et contrôle qualité des données collectées

In [2]:
import json
import math
from datetime import datetime, timezone

HOTELS_RAW_FILE = RAW_DIR / "hotels_booking.json"
hotels = pd.DataFrame(json.loads(HOTELS_RAW_FILE.read_text(encoding="utf-8")))

# Date de collecte : champ du spider s'il existe, sinon date de modification du fichier
if "collected_at" not in hotels.columns:
    mtime = datetime.fromtimestamp(HOTELS_RAW_FILE.stat().st_mtime, tz=timezone.utc)
    hotels["collected_at"] = mtime.isoformat(timespec="seconds")

print(f"{len(hotels)} hôtels collectés pour {hotels['city_id'].nunique()} destinations")
print("Hôtels par destination :", sorted(hotels.groupby("city_id").size().unique()))
print("Types d'établissement  :", hotels["accommodation_type_id"].value_counts().to_dict())
print("\nValeurs manquantes :")
print(hotels.isna().sum()[lambda s: s > 0])

700 hôtels collectés pour 35 destinations
Hôtels par destination : [np.int64(20)]
Types d'établissement  : {204: 698, 218: 2}

Valeurs manquantes :
distance_to_center    60
review_score           1
star_rating           68
dtype: int64


In [3]:
# Doublons : un même hôtel peut être proposé pour deux destinations voisines
in_same_city = hotels.duplicated(subset=["city_id", "hotel_id"]).sum()
shared = hotels[hotels.duplicated(subset="hotel_id", keep=False)].sort_values("hotel_id")
print("Doublons dans une même destination :", in_same_city)
print("Hôtels rattachés à plusieurs destinations :", shared["hotel_id"].nunique())
shared[["hotel_id", "hotel_name", "city_name", "hotel_city"]]

Doublons dans une même destination : 0
Hôtels rattachés à plusieurs destinations : 7


,hotel_id,hotel_name,city_name,hotel_city
205,51314,Hotel Le Colombier,Colmar,Colmar
238,51314,Hotel Le Colombier,Eguisheim,Colmar
640,51487,"Hôtel du Palais Biarritz, in The Unbound Colle...",Biarritz,Biarritz
679,51487,"Hôtel du Palais Biarritz, in The Unbound Colle...",Bayonne,Biarritz
229,57310,L'Esquisse Hotel & Spa Colmar - MGallery Colle...,Eguisheim,Colmar
200,57310,L'Esquisse Hotel & Spa Colmar - MGallery Colle...,Colmar,Colmar
232,245664,Relais & Châteaux La Maison Des Têtes,Eguisheim,Colmar
202,245664,Relais & Châteaux La Maison Des Têtes,Colmar,Colmar
201,1691123,Colmar Hotel,Colmar,Colmar
231,1691123,Colmar Hotel,Eguisheim,Colmar


In [4]:
# Plausibilité des valeurs numériques
hotels[["review_score", "review_count", "star_rating", "latitude", "longitude"]].describe().round(2)

,review_score,review_count,star_rating,latitude,longitude
count,699.00,700.00,632.00,700.00,700.00
mean,8.62,1288.33,3.37,45.84,3.40
std,0.52,1209.10,0.85,2.55,2.90
min,5.30,0.00,1.00,42.48,-2.26
25%,8.40,463.00,3.00,43.49,1.36
50%,8.70,936.50,3.00,45.19,4.36
75%,8.90,1719.75,4.00,48.55,5.71
max,9.70,9061.00,5.00,50.65,7.80


## 9. Distance entre chaque hôtel et sa destination

Le texte de distance fourni par Booking (« 300 m du centre », « 2,1 km de : Collioure ») est hétérogène et manquant
pour 60 hôtels. On le remplace par une **distance calculée** à partir des coordonnées GPS (formule de **haversine** :
distance « à vol d'oiseau » sur une sphère), **complète et comparable** d'une destination à l'autre.

In [5]:
def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Distance à vol d'oiseau (km) entre deux points GPS."""
    earth_radius_km = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi, d_lambda = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2
    return 2 * earth_radius_km * math.asin(math.sqrt(a))


cities = pd.read_csv(RAW_DIR / "cities.csv")[["city_id", "latitude", "longitude"]]
hotels = hotels.merge(cities, on="city_id", suffixes=("", "_city"), validate="many_to_one")
hotels["distance_km"] = hotels.apply(
    lambda row: haversine_km(row["latitude_city"], row["longitude_city"], row["latitude"], row["longitude"]), axis=1
).round(1)

print("Distance hôtel ↔ destination (km) :")
print(hotels["distance_km"].describe().round(1))
print("\nDestinations dont certains hôtels sont à plus de 20 km :")
hotels[hotels["distance_km"] > 20].groupby("city_name")["distance_km"].agg(["count", "max"])

Distance hôtel ↔ destination (km) :
count    700.0
mean       3.7
std        6.4
min        0.0
25%        0.5
50%        1.3
75%        3.7
max       43.9
Name: distance_km, dtype: float64

Destinations dont certains hôtels sont à plus de 20 km :


,count,max
city_name,,
Ariege,11,43.5
Gorges du Verdon,5,43.9
St Malo,5,28.3


### Analyse
- Les hôtels éloignés concernent surtout **Ariege** (un département entier) et les **Gorges du Verdon** (un site
  de plusieurs dizaines de km) : c'est cohérent avec la nature de ces destinations (cf. étape 1).
- Pour les autres, Booking élargit la recherche aux environs quand la commune compte peu d'hôtels
  (ex. : Dinan ou Dol-de-Bretagne pour Saint-Malo).
- Les hôtels sont **tous conservés** dans les données (rien n'est perdu) ; la colonne `distance_km` permettra de
  filtrer au moment de l'analyse (Top 20 des hôtels, étape 6).

## 10. Carte de contrôle : hôtels des 5 meilleures destinations (météo)

In [6]:
import plotly.express as px

weather_summary = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "weather_summary.csv")
top5_ids = weather_summary.nsmallest(5, "weather_rank")["city_id"]
sample = hotels[hotels["city_id"].isin(top5_ids)]

fig = px.scatter_map(
    sample, lat="latitude", lon="longitude", color="city_name",
    hover_name="hotel_name", hover_data={"review_score": True, "distance_km": True, "latitude": False, "longitude": False},
    zoom=6, height=600, map_style="open-street-map",
    title="Hôtels collectés pour les 5 meilleures destinations météo (contrôle)",
)
fig.show()

## 11. Enregistrement des données nettoyées

On conserve les champs utiles, avec des types propres. Le fichier JSON brut reste dans `data/raw/` ;
la version nettoyée va dans `data/processed/` (zone `processed/` du Data Lake à l'étape 4).

Choix de nettoyage :
- `distance_to_center` (texte Booking, incomplet) est remplacé par `distance_km` (calculée) ;
- les coordonnées de la destination ajoutées pour le calcul (`latitude_city`, `longitude_city`) sont retirées ;
- les valeurs manquantes (`star_rating` : hôtel non classé ; `review_score` : hôtel sans note) sont **laissées vides** :
  les remplacer par une moyenne reviendrait à **inventer** une note ou une catégorie.

In [7]:
COLUMNS = [
    "city_id", "hotel_id", "search_position", "hotel_name", "hotel_url", "accommodation_type_id",
    "latitude", "longitude", "address", "hotel_city", "distance_km",
    "review_score", "review_count", "star_rating", "description", "collected_at",
]
hotels_clean = hotels[COLUMNS].astype({"city_id": int, "hotel_id": int, "search_position": int,
                                       "accommodation_type_id": int, "review_count": int})
hotels_clean["star_rating"] = hotels_clean["star_rating"].astype("Int64")  # entier pouvant être vide

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
HOTELS_FILE = PROCESSED_DIR / "hotels.csv"
hotels_clean.to_csv(HOTELS_FILE, index=False, encoding="utf-8")
print(f"{len(hotels_clean)} lignes enregistrées dans {HOTELS_FILE}")
hotels_clean.head()

700 lignes enregistrées dans c:\Users\frede\OneDrive\Documents\_Jedha\2- dsfs-od-15 - Fullstack\Final Projects - dsfs-od-15\B1 - Kayak\kayak-plan-your-trip\data\processed\hotels.csv


,city_id,hotel_id,search_position,hotel_name,hotel_url,accommodation_type_id,latitude,longitude,address,hotel_city,distance_km,review_score,review_count,star_rating,description,collected_at
0,1,57176,1,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,204,48.614247,-1.510545,La Caserne,Le Mont Saint Michel,2.4,8.4,3255,4,Installé dans des espaces verts à seulement 2 ...,2026-11-05T09:05:26+00:00
1,1,53560,2,Le Relais Du Roy,https://www.booking.com/hotel/fr/le-relais-du-...,204,48.616263,-1.510906,La Caserne,Le Mont Saint-Michel,2.2,8.3,2534,3,Le Relais Du Roy est un hôtel 3 étoiles situé ...,2026-11-05T09:05:26+00:00
2,1,230795,3,Le Relais Saint Michel,https://www.booking.com/hotel/fr/le-relais-sai...,204,48.617587,-1.510396,La Caserne,Le Mont-Saint-Michel,2.0,8.0,3658,4,Le Relais Saint Michel vous accueille face à l...,2026-11-05T09:05:26+00:00
3,1,171494,4,Hotel Gabriel,https://www.booking.com/hotel/fr/hotel-gabriel...,204,48.615381,-1.510710,Route du Mont Saint Michel,Le Mont Saint Michel,2.3,8.1,2369,3,Hotel Gabriel is located 1.6 Km from Mont Sain...,2026-11-05T09:05:26+00:00
4,1,275706,5,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html,204,48.614700,-1.509617,La Caserne,Le Mont Saint Michel,2.4,8.1,5677,2,L’Hotel Vert vous propose des chambres décorée...,2026-11-05T09:05:26+00:00
